In [ ]:
# ==========================================
# Exp-9: Text Generation using LSTM Networks
# ==========================================

# Step 1: Import Required Libraries
import tensorflow as tf
import numpy as np

# Step 2: Load and Preprocess the Dataset
file_path = tf.keras.utils.get_file(
    "shakespeare.txt",
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)

with open(file_path, "r", encoding="utf-8") as file:
    text = file.read().lower()

chars = sorted(set(text))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for i, c in enumerate(chars)}

print("Total characters:", len(text))
print("Unique characters:", len(chars))

# Step 3: Create Input and Output Sequences
seq_len = 40
num_sequences = 100000

# Encode the text once
encoded_text = np.array(
    [c2i[c] for c in text],
    dtype=np.int32
)

# Select sequences spread across the entire text
starts = np.linspace(
    0,
    len(encoded_text) - seq_len - 1,
    num_sequences,
    dtype=np.int32
)

offsets = np.arange(seq_len)

X = encoded_text[starts[:, None] + offsets]
y = encoded_text[starts + seq_len]

print("Input shape:", X.shape)
print("Target shape:", y.shape)

# Step 4: Build the LSTM Model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(seq_len,)),
    tf.keras.layers.Embedding(
        input_dim=len(chars),
        output_dim=64
    ),
    tf.keras.layers.LSTM(96),
    tf.keras.layers.Dense(
        len(chars),
        activation="softmax"
    )
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy"
)

model.summary()

# Step 5: Train the LSTM Model
history = model.fit(
    X,
    y,
    batch_size=256,
    epochs=3,
    shuffle=True,
    verbose=1
)

# Step 6: Define the Text Generation Function
def generate(seed, length=300, temperature=0.7, top_k=5):
    seed = seed.lower()

    sequence = [c2i[c] for c in seed if c in c2i]

    if len(sequence) < seq_len:
        sequence = [c2i[" "]] * (seq_len - len(sequence)) + sequence

    generated_text = seed

    for _ in range(length):
        input_sequence = np.array(
            sequence[-seq_len:],
            dtype=np.int32
        ).reshape(1, seq_len)

        # Predict the next character
        probabilities = model(
            input_sequence,
            training=False
        ).numpy()[0]

        # Apply temperature
        logits = np.log(probabilities + 1e-8) / temperature

        # Keep only the most likely characters
        k = min(top_k, len(logits))
        top_indices = np.argpartition(logits, -k)[-k:]

        filtered_logits = logits[top_indices]
        filtered_logits -= np.max(filtered_logits)

        top_probabilities = np.exp(filtered_logits)
        top_probabilities /= np.sum(top_probabilities)

        # Sample the next character
        next_index = np.random.choice(
            top_indices,
            p=top_probabilities
        )

        sequence.append(int(next_index))
        generated_text += i2c[int(next_index)]

    return generated_text

# Step 7: Generate and Display Text
seed_text = "shall i compare thee to a summer's day?\n"

generated_text = generate(
    seed_text,
    length=300,
    temperature=0.7,
    top_k=5
)

print("\nGenerated Text:\n")
print(generated_text)

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Total characters: 1115394
Unique characters: 39
Input shape: (100000, 40)
Target shape: (100000,)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 64)         │         2,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 96)             │        61,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 39)             │         3,783 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,103 (266.03 KB)

 Trainable params: 68,103 (266.03 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 56s 137ms/step - loss: 2.6761
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 53s 136ms/step - loss: 2.2773
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 53s 137ms/step - loss: 2.1468

Generated Text:

shall i compare thee to a summer's day?

carese:
the me the the sonend the mome and as thow whis to mit i hersed in thee the that tere sour in and to the serontir on wome the this he whe with and me his that wher and and to le he shount me tath the me whit will thou the shat the prome thir he the thour how and he the her weard to share an
